# Data Checks

Slow diagnostic cells that scan the full dataset. Run these infrequently — not part of the main analysis pipeline.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

## Temporal coverage: sample vs full dataset

In [ ]:
def stream_dates(path):
    """Read only start_time from each record."""
    dates = []
    with open(path) as f:
        for line in f:
            dates.append(json.loads(line)['start_time'][:10])
    return pd.to_datetime(dates).to_series().dt.to_period('W').value_counts().sort_index()

sample_weekly = stream_dates('data/collected_matches_sample.jsonl')
full_weekly   = stream_dates('data/collected_matches.jsonl')

sample_aligned = sample_weekly.reindex(full_weekly.index, fill_value=0)
x      = range(len(full_weekly))
labels = [str(p) for p in full_weekly.index]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].bar(x, full_weekly.values, color='steelblue', width=0.8)
axes[0].set_title(f'Full dataset — {full_weekly.sum():,} matches by week')
axes[0].set_ylabel('Matches')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x, sample_aligned.values, color='tomato', width=0.8)
axes[1].set_title(f'Sample — {sample_aligned.sum():,} matches by week (same x-axis)')
axes[1].set_ylabel('Matches')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Temporal coverage: sample vs full dataset', fontsize=13)
plt.tight_layout()
plt.show()

## Temporal coverage: sample vs full + gap dataset

In [ ]:
def stream_dates_combined(*paths):
    """Stream start_time from one or more JSONL files, skip missing files."""
    dates = []
    for path in paths:
        try:
            with open(path) as f:
                for line in f:
                    try:
                        dates.append(json.loads(line)['start_time'][:10])
                    except (json.JSONDecodeError, KeyError):
                        pass
        except FileNotFoundError:
            print(f'  {path} not found — skipping')
    return pd.to_datetime(dates).to_series().dt.to_period('W').value_counts().sort_index()

combined_weekly  = stream_dates_combined('data/collected_matches.jsonl', 'data/collected_matches_gap.jsonl')
sample_weekly_c  = stream_dates_combined('data/collected_matches_sample.jsonl')
sample_aligned_c = sample_weekly_c.reindex(combined_weekly.index, fill_value=0)

x      = range(len(combined_weekly))
labels = [str(p) for p in combined_weekly.index]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].bar(x, combined_weekly.values, color='steelblue', width=0.8)
axes[0].set_title(f'Full + gap dataset — {combined_weekly.sum():,} matches by week')
axes[0].set_ylabel('Matches')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x, sample_aligned_c.values, color='tomato', width=0.8)
axes[1].set_title(f'Sample — {sample_aligned_c.sum():,} matches (same x-axis)')
axes[1].set_ylabel('Matches')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Temporal coverage: sample vs combined dataset', fontsize=13)
plt.tight_layout()
plt.show()

## Match counts per script window

Scans the full dataset to verify each week window has the expected ~6,000 matches.

In [ ]:
WEEK_WINDOWS = [
    ("2025-W32", 38_408_619, 38_956_500),
    ("2025-W33", 38_956_501, 39_504_382),
    ("2025-W34", 39_504_383, 40_052_264),
    ("2025-W35", 40_052_265, 40_611_632),
    ("2025-W36", 40_611_634, 41_404_909),
    ("2025-W37", 41_404_910, 42_198_185),
    ("2025-W38", 42_198_186, 42_991_461),
    ("2025-W39", 42_991_462, 43_779_832),
    ("2025-W40", 43_779_833, 44_409_896),
    ("2025-W41", 44_409_897, 45_039_960),
    ("2025-W42", 45_039_961, 45_669_793),
    ("2025-W43", 45_669_794, 46_299_627),
    ("2025-W44", 46_299_638, 46_724_879),
    ("2025-W45", 46_724_880, 47_150_121),
    ("2025-W46", 47_150_122, 47_675_083),
    ("2025-W47", 47_675_084, 48_200_047),
    ("2025-W48", 48_200_048, 48_651_271),
    ("2025-W49", 48_651_272, 49_102_495),
    ("2025-W50", 49_102_496, 49_555_710),
    ("2025-W51", 49_555_711, 50_008_927),
    ("2026-W01", 50_008_931, 51_039_637),
    ("2026-W02", 51_039_638, 52_070_344),
    ("2026-W03", 52_070_345, 53_101_051),
    ("2026-W04", 53_101_052, 54_144_829),
    ("2026-W05", 54_144_839, 56_445_934),
    ("2026-W06", 56_445_935, 58_746_030),
    ("2026-W07", 58_746_031, 61_047_126),
    ("2026-W08", 61_047_127, 63_188_639),
    ("2026-W09", 63_188_643, 65_490_120),
    ("2026-W10", 65_490_121, 67_791_598),
    ("2026-W11", 67_791_599, 70_002_386),
    ("2026-W12", 70_002_387, 72_213_175),
    ("2026-W13", 72_213_176, 73_196_674),
    ("2026-W14", 73_196_675, 74_721_873),
]

window_counts = {label: 0 for label, _, _ in WEEK_WINDOWS}

with open('data/collected_matches.jsonl') as f:
    for line in f:
        try:
            mid = json.loads(line)['match_id']
            for label, lo, hi in WEEK_WINDOWS:
                if lo <= mid <= hi:
                    window_counts[label] += 1
                    break
        except (json.JSONDecodeError, KeyError):
            pass

counts_df = pd.DataFrame([
    {'week': label, 'lo': lo, 'hi': hi, 'count': window_counts[label]}
    for label, lo, hi in WEEK_WINDOWS
])
display(counts_df.to_string(index=False))

## Gap file diagnostic

Check how many matches `collected_matches_gap.jsonl` collected per week.

In [ ]:
try:
    gap_rows = []
    with open('data/collected_matches_gap.jsonl') as f:
        for line in f:
            try:
                rec = json.loads(line)
                gap_rows.append({'start_time': rec['start_time'][:10], 'match_id': rec['match_id']})
            except (json.JSONDecodeError, KeyError):
                pass
    gap_df = pd.DataFrame(gap_rows)
    gap_df['start_time'] = pd.to_datetime(gap_df['start_time'])
    print(f'Total matches in gap file: {len(gap_df)}')
    display(gap_df.groupby(gap_df['start_time'].dt.to_period('W')).agg(
        matches=('match_id', 'count'),
        min_id=('match_id', 'min'),
        max_id=('match_id', 'max'),
    ))
except FileNotFoundError:
    print('Gap file not found.')